# 01 - Quick Validation of All Datasets

A quick pass (not a full EDA) over everything in `datasets/raw/`: CRS, resolution, extent, nodata, and basic value ranges. The goal is just to catch broken files before moving on to preprocessing (clip/reproject/grid).

A proper EDA (distributions, feature correlations, spatial patterns) makes more sense once everything's been harmonized onto the same 1km grid - see the project notes.

*Duplicate copy kept in `evaluations/` so every dataset notebook can be found in one place. The maintained source is **`scripts/preprocessing/01_validate_datasets.ipynb`** — edit it there; this copy is regenerated from it, not edited directly.*

In [1]:
# add scripts/download to sys.path so we can `from common import ...`
# (common.py is shared between the download and preprocessing notebooks, not duplicated)
import os, sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())
NOTEBOOK_DIR = PROJECT_ROOT / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\preprocessing


## 1. Scan every GeoTIFF

In [2]:
import zipfile
from pathlib import Path

import geopandas as gpd
import pandas as pd
import rasterio

from common import RAW_DIR

log_rows = []


def summarize_raster(path: Path) -> dict:
    with rasterio.open(path) as src:
        band1 = src.read(1, masked=True)
        valid = band1.compressed()
        bounds = src.bounds
        width_deg = bounds.right - bounds.left
        return {
            "file": str(path.relative_to(RAW_DIR)),
            "dataset": path.parent.name,
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "crs": str(src.crs),
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "res_m_or_deg": round(src.res[0], 6),
            "bounds": tuple(round(b, 3) for b in bounds),
            "pct_masked": round(float(band1.mask.mean()) * 100, 1) if hasattr(band1, "mask") and band1.mask is not False else 0.0,
            "min": float(valid.min()) if valid.size else None,
            "max": float(valid.max()) if valid.size else None,
            "mean": round(float(valid.mean()), 3) if valid.size else None,
            "global_extent": width_deg > 300,  # rough heuristic: ~world-wide extent
        }

In [3]:
tif_files = sorted(RAW_DIR.rglob("*.tif"))
print(f"Found {len(tif_files)} .tif files under datasets/raw/")

for f in tif_files:
    try:
        log_rows.append(summarize_raster(f))
    except Exception as e:  # noqa: BLE001
        log_rows.append({"file": str(f.relative_to(RAW_DIR)), "dataset": f.parent.name, "error": str(e)})

df = pd.DataFrame(log_rows)

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
df

Found 113 .tif files under datasets/raw/


,file,dataset,width,height,bands,crs,dtype,nodata,res_m_or_deg,bounds,pct_masked,min,max,mean,global_extent
0,chirps\chirps-v2.0.2020.01.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,1774.078735,-7255.641,True
1,chirps\chirps-v2.0.2020.02.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,966.216248,-7256.900,True
2,chirps\chirps-v2.0.2020.03.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,1168.165771,-7254.116,True
3,chirps\chirps-v2.0.2020.04.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,2103.821289,-7256.006,True
4,chirps\chirps-v2.0.2020.05.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,2122.798096,-7255.292,True
5,chirps\chirps-v2.0.2020.06.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,1439.253296,-7253.685,True
6,chirps\chirps-v2.0.2020.07.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,1501.271484,-7251.220,True
7,chirps\chirps-v2.0.2020.08.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,1705.191162,-7251.484,True
8,chirps\chirps-v2.0.2020.09.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,2291.218994,-7254.502,True
9,chirps\chirps-v2.0.2020.10.tif,chirps,7200,2000,1,EPSG:4326,float32,NaN,0.050000,"(-180.0, -50.0, 180.0, 50.0)",0.0,-9999.000000,2178.286377,-7256.167,True


## 2. Summary & things worth a second look

In [4]:
print("=== Per-dataset summary ===")
print(df.groupby("dataset").agg(n_files=("file", "count"), n_crs=("crs", "nunique")))

print("\n=== Files still at global extent (not yet clipped to Banten) ===")
global_files = df[df.get("global_extent", False) == True]
print(global_files[["dataset", "file", "bounds"]] if not global_files.empty else "(none)")

print("\n=== Files with > 50% masked/nodata ===")
high_nodata = df[df.get("pct_masked", 0) > 50]
print(high_nodata[["dataset", "file", "pct_masked"]] if not high_nodata.empty else "(none)")

report_path = RAW_DIR / "_validation_report.csv"
df.to_csv(report_path, index=False)
print(f"\nFull report saved to {report_path}")

=== Per-dataset summary ===
              n_files  n_crs
dataset                     
chirps             60      1
dem                 1      1
gaez_v5             4      1
sentinel2gee       20      1
soilgrids           7      1
worldclim          19      1
worldcover          2      1

=== Files still at global extent (not yet clipped to Banten) ===
          dataset                                               file                                      bounds
0          chirps                     chirps\chirps-v2.0.2020.01.tif                (-180.0, -50.0, 180.0, 50.0)
1          chirps                     chirps\chirps-v2.0.2020.02.tif                (-180.0, -50.0, 180.0, 50.0)
2          chirps                     chirps\chirps-v2.0.2020.03.tif                (-180.0, -50.0, 180.0, 50.0)
3          chirps                     chirps\chirps-v2.0.2020.04.tif                (-180.0, -50.0, 180.0, 50.0)
4          chirps                     chirps\chirps-v2.0.2020.05.tif            

## 3. Check the vector data (admin boundary)

In [5]:
aoi_path = RAW_DIR / "boundaries" / "aoi_banten.geojson"
gdf = gpd.read_file(aoi_path)
geom = gdf.geometry.iloc[0]
print("aoi_banten.geojson")
print("  CRS:", gdf.crs)
print("  geometry type:", geom.geom_type)
print("  valid geometry:", geom.is_valid)
print("  bounds (west, south, east, north):", gdf.total_bounds)

aoi_banten.geojson
  CRS: EPSG:4326
  geometry type: MultiPolygon
  valid geometry: True
  bounds (west, south, east, north): [105.0995575  -7.0367223 106.7800127  -5.8077777]


## 4. Check the Zenodo datasets still sitting as .zip

In [6]:
zip_files = [
    RAW_DIR / "global_crop_suitability_2024" / "2024.zip",
    RAW_DIR / "cropsuite" / "suitability_without_climate_variability.zip",
]

for zp in zip_files:
    if not zp.exists():
        print(f"{zp} - NOT FOUND")
        continue
    with zipfile.ZipFile(zp) as zf:
        names = zf.namelist()
    size_gb = zp.stat().st_size / 1024**3
    print(f"{zp.relative_to(RAW_DIR)} ({size_gb:.2f} GB compressed) - {len(names)} entries, e.g.:")
    for n in names[:5]:
        print("   ", n)
    print("  -> not extracted yet, needs unpacking before the preprocessing stage")

global_crop_suitability_2024\2024.zip (3.10 GB compressed) - 17 entries, e.g.:
    Barley.tif
    Cassava.tif
    Cotton.tif
    Maize.tif
    Millet.tif
  -> not extracted yet, needs unpacking before the preprocessing stage
cropsuite\suitability_without_climate_variability.zip (1.12 GB compressed) - 2820 entries, e.g.:
    suitability_without_climate_variability/
    __MACOSX/._suitability_without_climate_variability
    suitability_without_climate_variability/sweetpotato/
    __MACOSX/suitability_without_climate_variability/._sweetpotato
    suitability_without_climate_variability/avocado/
  -> not extracted yet, needs unpacking before the preprocessing stage
